## 4.2. Compute Region Temperature

This notebook shows the temperature in the region directly below the tree was computed

In [ ]:
import os, shutil, cv2,datetime, torch, torchvision.transforms, json, torchvision, copy, time, open_clip, sys,PIL.Image
os.chdir('/home/klimenko/CoolingMachines/explore_flir')
import matplotlib.pyplot as plt
print(os.getcwd())
import pandas as pd
import numpy as np
import matplotlib.colors as mcolors
from PIL import Image
import time
import numpy as np
from scipy.ndimage import zoom
from tqdm import tqdm
import matplotlib.pyplot as plt
from PIL import Image,ImageFile
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import torch
from torch.utils.data import random_split
import torch.nn as nn
import pandas as pd
from collections import Counter
from mit_semseg.config import cfg
from mit_semseg.dataset import TestDataset
from mit_semseg.models import ModelBuilder, SegmentationModule
from transformers import CLIPSegProcessor, CLIPSegForImageSegmentation

####################################
# ADE20K-based Scene Parsing Segmentation to Extract Facade
###################################

# This module (mit_semseg) is identical to https://pypi.org/project/mit-semseg/
# Obtainted from:  https://github.com/CSAILVision/sceneparsing

# Reference: Semantic Understanding of Scenes through ADE20K Dataset. 
# B. Zhou, H. Zhao, X. Puig, T. Xiao, S. Fidler, A. Barriuso and A. Torralba.
# International Journal on Computer Vision (IJCV), 2018. (https://arxiv.org/pdf/1608.05442.pdf)

net_encoder = ModelBuilder.build_encoder(arch='resnet50dilated',fc_dim=2048,weights='ckpt/ade20k-resnet50dilated-ppm_deepsup/encoder_epoch_20.pth')
net_decoder = ModelBuilder.build_decoder(arch='ppm_deepsup',fc_dim=2048,num_class=150,weights='ckpt/ade20k-resnet50dilated-ppm_deepsup/decoder_epoch_20.pth',use_softmax=True)
crit = torch.nn.NLLLoss(ignore_index=-1)
segmentation_module = SegmentationModule(net_encoder, net_decoder, crit)
segmentation_module.eval()
device = torch.device("cuda:0")
segmentation_module.to(device)
pil_to_tensor = torchvision.transforms.Compose([torchvision.transforms.ToTensor(),torchvision.transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])  ])


####################################
# ClipSeg
###################################
# Source: https://huggingface.co/docs/transformers/model_doc/clipseg
# Reference: Luddecke, Timo, and Alexander Ecker. "Image segmentation using text and image prompts."
# Proceedings of the IEEE/CVF conference on computer vision and pattern recognition. 2022.


from transformers import CLIPSegProcessor, CLIPSegForImageSegmentation
processor = CLIPSegProcessor.from_pretrained("CIDAS/clipseg-rd64-refined")
model = CLIPSegForImageSegmentation.from_pretrained("CIDAS/clipseg-rd64-refined")

In [ ]:
def segment_image(path):
    pil_image = PIL.Image.open(path).convert('RGB')
    width, height = pil_image.size
    resized_pil_image = pil_image
    img_data = pil_to_tensor(resized_pil_image)
    singleton_batch = {'img_data': img_data[None].to(device)}
    output_size = img_data.shape[1:]
    with torch.no_grad():
        scores = segmentation_module(singleton_batch, segSize=output_size)
    _, pred = torch.max(scores, dim=1)
    pred = pred.cpu()[0].numpy()
    return pred

def obtain_temp_scale(img_fresh, scalebar):
    linear_interpolation_array_temp = np.linspace(0,40, len(scalebar)).astype(float)
    img_temp = np.interp(img_fresh[:,:,0].astype(np.float32),scalebar[:,0], linear_interpolation_array_temp)
    return img_temp

In [ ]:
result = pd.read_csv('/home/klimenko/CoolingMachines/explore_flir/dataset/AMSTERDAM_FLIR_DATASET/all_23.csv')

In [ ]:
visual_path = list(result['visual_path'])[89]
thermal_path = list(result['thermal_path'])[89]


def func_region(visual_path,thermal_path, xyxy_string):

    pred = segment_image(visual_path)
    thermal_img_fresh = cv2.imread(thermal_path)

    file_name, file_extension = os.path.splitext(os.path.basename(thermal_path))
    new_file_name = file_name + file_extension.lower().replace(".jpeg", ".jpg") + ".npy"
    scalebar = np.load('dataset/BOSTON_FLIR_DATASET/all/watermarked/'+new_file_name)
    thermal = obtain_temp_scale(thermal_img_fresh, scalebar) 


    
    
    visual = cv2.imread(visual_path)
    b = visual.mean(axis=2)
    
    # We threshold all pixels darker than a boundary value so that regions in shade do not affect calculation
    light_threshold_mask = np.where(b[:,:]<45, 1, 0)
    light_threshold_mask = cv2.resize(light_threshold_mask.astype(np.float32), (480, 640))
    
    
    s = xyxy_string
    xyxy = list(map(int, s.strip('[]').split(', ')))
    zeros_array = np.zeros_like(visual)
    
    # 'Region' is a region under the tree drawn as a line with a rounded edge to imitate a circle. The vertical diameter is 1/3 of the tree's height.
    qwe2 = cv2.line(zeros_array, (xyxy[0],xyxy[3]), (xyxy[2],xyxy[3]), (255, 255, 255), thickness=int((xyxy[2]-xyxy[0])/3), lineType=cv2.LINE_AA)
    region_mask = qwe2[:,:,0]/255
    
     
    region_mask = cv2.resize(region_mask.astype(np.float32), (480, 640))
    
    # 'Other' objects are surfaces on the ground (grass, road and asphalt), which are directly under the tree
    other_mask = np.isin(pred, [9,6,11]).astype(np.float32)#
    other_mask = cv2.resize(other_mask.astype(np.float32), (480, 640))

    other_mask = other_mask[40:, :]
    region_mask = region_mask[40:, :]
    light_threshold_mask = light_threshold_mask[40:, :]
    thermal = thermal[:-40, :]

    # The region under the tree is obtained by filtering of the thermal mask (thermal) to 
    # include the "region" below a tree (region_mask),
    # filtered by presence on the asphalt/grass (other_mask),
    # not in dark pixels (1-light_threshold_mask)
    region = other_mask * thermal[:,:] * region_mask * (1-light_threshold_mask) 
    region_temperature = np.nanmean(region[region > 10])
    # The area not in this region is similar except that it is a negative of the region_mask
    outside = other_mask * thermal[:,:] * (1-region_mask) * (1-light_threshold_mask)
    outside_temperature = np.nanmean(outside[outside > 10]) 
    
    return region_temperature, outside_temperature

In [ ]:
# Run Calculation Function throughout the dataset

import pandas as pd
import cv2

df = pd.read_csv('/home/klimenko/CoolingMachines/explore_flir/dataset/AMSTERDAM_FLIR_DATASET/all_23.csv')


# Initialize empty lists to hold the computed values
tree_temperature_values = []
other_temperature_values = []

# Loop over the rows in the DataFrame and apply the function
total_rows = len(df)
for idx, row in df.iterrows():
    
    try:
        # Extract values using the image path and info
        tree_temperature, other_temperature = func_region(row['visual_path'], row['thermal_path'], row['xyxy'])
        print(tree_temperature, other_temperature)

        # Append the results to the respective lists
        tree_temperature_values.append(tree_temperature)
        other_temperature_values.append(other_temperature)

        # Print progress manually
        if (idx + 1) % 10 == 0 or (idx + 1) == total_rows:
            print(f"Processed {idx + 1}/{total_rows} rows____________________")
            
    except:
        print('F')
        tree_temperature_values.append(np.nan)
        other_temperature_values.append(np.nan)

# Add the computed values as new columns in the DataFrame
df['tree_region_temperature_scaled2_road_grass_noshadow'] = tree_temperature_values
df['outside_region_temperature_scaled2_road_grass_noshadow'] = other_temperature_values


In [ ]:
df.to_csv('/home/klimenko/CoolingMachines/explore_flir/dataset/AMSTERDAM_FLIR_DATASET/all_23.csv')